In [ ]:
"""Sub-Holding Keys

Demonstration of how to set up and use sub-holding keys

Attributes
----------
properties
sub-holding keys
cocoon - seed_data
holdings
prorated
"""

In [ ]:
# %load_ext lab_black
# %load_ext nb_black

# 1. Sub-Holding Keys

This notebook demonstrates LUSID's [Sub-holding Keys](https://support.finbourne.com/what-are-subholding-keys) (or SHKs). The core idea with `Sub-holding Keys` - they allow you to bucket your `holding` in one instrument (or [LUID](https://support.finbourne.com/what-is-a-lusid-unique-identifier-luid)) into different groups. For example, in this notebook we have a `Sub-Holding Key` of <i>strategy</i> which is used to tag transactions on the same instrument following different investment strategies. Then in the `holdings` report you can see the position split-out into two buckets. However, this is just one sample implementation of `Sub-Holding Keys`. You are allowed use <u>any</u> pre-defined transaction property as a `Sub-Holding Key`. 

# 2. Setup LUSID

In [ ]:
# Import general purpose packages
import os
import json
from datetime import datetime, timedelta
import pytz

# Import lusid specific packages
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
from finbourne.sdk.exceptions import ApiException
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.cocoon.seed_sample_data import seed_data
from finbourne_sdk_utils.cocoon.utilities import create_scope_id

# Import data wrangling packages
import pandas as pd

pd.set_option("display.max_columns", None)

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook",
)

Load a mapping file for DataFrame headers for the `build transaction` and `get holdings` response.

In [ ]:
with open(r"config/build_transactions_mapping.json") as mappings_file:
    build_transactions_json_mapping = json.load(mappings_file)

with open(r"config/get_holdings_mapping.json") as mappings_file:
    get_holdings_json_mapping = json.load(mappings_file)

Define our transaction portfolios API

In [ ]:
transaction_portfolios_api = api_factory.build(lu.TransactionPortfoliosApi)

# 3. Load Data

## 3.1 Declare a scope and load our CSV file

In [ ]:
# Create a new scope

scope = "notebook_shk1"
portfolio_code = "EQUITY_UK" + "_" + create_scope_id().replace("-", "")

In [ ]:
# Load a file of equity transactions

transactions_file = r"data/shk/equity_transactions.csv"
transactions_df = pd.read_csv(transactions_file)
transactions_df["portfolio_code"] = portfolio_code
transactions_df.tail(2)

## 3.2 Create a property for the new Sub-Holding key

The <b>strategy</b> will be used to create our `Sub-Holding Key` on the portfolio.

In [ ]:
domain = "Transaction"
scope = scope
prop_code = "strategy"

try:
    api_factory.build(lu.PropertyDefinitionsApi).create_property_definition(
        create_property_definition_request=models.CreatePropertyDefinitionRequest(
            domain=domain,
            scope=scope,
            code=prop_code,
            value_required=None,
            display_name="Investment strategy",
            data_type_id=models.ResourceId(scope="system", code="string"),
            life_time=None,
        )
    )

except ApiException as e:
    print(json.loads(e.body)["title"])

## 3.3 Load default transactions into a new scope

The portfolio is created with the new `Sub-holding Key`.

In [ ]:
# Load portfolios, instruments, and transactions

seed_data_response = seed_data(
    api_factory,
    ["portfolios", "instruments", "transactions"],
    scope,
    transactions_df,
    "DataFrame",
    sub_holding_keys=[f"Transaction/{scope}/strategy"],
)

# 4. Lets check our holdings

We have can see that the one Tesco instrument (with the same LUID) is bucketed under two different `Sub-Holding Keys` with the <b>strategy</b> label. There is also a seperate CCY_GBP cash line for tracking the cash in each SHK.

In [ ]:
response = transaction_portfolios_api.get_holdings(
    scope=scope,
    code=portfolio_code,
    property_keys=["Instrument/default/Name", "Instrument/default/ClientInternal"],
    filter="properties.Instrument/default/Name in ('Tesco', 'CCY_GBP')",
)

holdings_df = lusid_response_to_data_frame(
    response, rename_properties=True, column_name_mapping=get_holdings_json_mapping
)

holdings_df

# 5. Book a prorated transaction across two SHKs

In this section we book one transaction and do a prorated allocation across two SHKs.

## 5.1 Create a new transaction type

First we create a new "Buy" transaction with the prorata configuration. Specifically we add a `AllocationMethod=Prorated` property.

In [ ]:
transaction_configuration_api = api_factory.build(lu.TransactionConfigurationApi)

# Add default side definitions to the non-default transaction type scope.
# If working in the default scope, these side definitions are set by default so, unless these sides have been removed, this setting of sides can be skipped.
default_side_definitions = [
    models.SidesDefinitionRequest(
        side="Side1", 
        side_request=models.SideDefinitionRequest(
            security="Txn:LusidInstrumentId",
            currency="Txn:TradeCurrency",
            rate="Txn:TradeToPortfolioRate",
            units="Txn:Units",
            amount="Txn:TradeAmount")),
    models.SidesDefinitionRequest(
        side="Side2", 
        side_request=models.SideDefinitionRequest(
            security="Txn:SettleCcy",
            currency="Txn:SettlementCurrency",
            rate="SettledToPortfolioRate",
            units="Txn:TotalConsideration",
            amount="Txn:TotalConsideration"))
]

transaction_configuration_api.set_side_definitions(default_side_definitions, scope = scope)

# Add default transaction types
default_transaction_mapping=open('data/default_transaction_mapping.json').read()
default_transaction_mapping = json.loads(default_transaction_mapping)

def map_properties(properties):
    return {property["key"]: models.PerpetualProperty(key=property["key"], value=models.PropertyValue(label_value=property["value"])) for property in properties}
def map_alias(alias):
    return models.TransactionTypeAlias(type=alias["type"], description=alias["description"], transaction_class=alias["transactionClass"], transaction_roles=alias["transactionRoles"])
def map_movement(movement):
    return models.TransactionTypeMovement(movement_types=movement["movementTypes"], side=movement["side"], direction=movement["direction"], properties=map_properties(movement["properties"]))
def map_transaction_type_request(transaction_type_request):
    return models.TransactionTypeRequest(
        aliases=[map_alias(alias) for alias in transaction_type_request["aliases"]],
        movements=[map_movement(movement) for movement in transaction_type_request["movements"]],
        properties=map_properties(transaction_type_request["properties"]))

for configuration in default_transaction_mapping:
    transaction_type_requests = [map_transaction_type_request(transaction_type_request) for transaction_type_request in configuration["transactionTypeRequests"]]
    
    # Call LUSID to set your configuration for our transaction types
    transaction_configuration_api.set_transaction_type_source(
        source=configuration["source"],
        transaction_type_request=transaction_type_requests,
        scope=scope
    )

# Prepare the Transaction Type model for new transaction type

new_transaction_config=models.TransactionTypeRequest(
    aliases=[
        models.TransactionTypeAlias(
            type="BuyProRated",
            description="An BuyProRated transaction type",
            transaction_class="default",
            transaction_roles="Longer",
        )
    ],
    movements=[
        models.TransactionTypeMovement(
            movement_types="StockMovement",
            side="Side1",
            direction=1,
            properties={},
            mappings=[],
        ),
        models.TransactionTypeMovement(
            movement_types="CashCommitment",
            side="Side2",
            direction=-1,
            properties={},
            mappings=[],
        )
    ],
    properties={"TransactionConfiguration/default/AllocationMethod": models.PerpetualProperty(
        key="TransactionConfiguration/default/AllocationMethod",
        value=models.PropertyValue(label_value="Prorated"))}
)

In [ ]:
# Upload the transaction type

new_txn_config = transaction_configuration_api.set_transaction_type(
    source="default",
    type="BuyProRated",
    scope=scope,
    transaction_type_request=new_transaction_config
)

# Update the transaction type scope of your portfolio 
patch_document = [
    {
        "value": scope,
        "path": "/transactiontypescope",
        "op": "add"
    }
]
patch_response = api_factory.build(lu.TransactionPortfoliosApi).patch_portfolio_details(
    scope=scope,
    code=portfolio_code,
    operation=patch_document)

## 5.2 Book a transaction

Next we book a Tesco PLC transaction for 6000 units. We already hold Tesco PLC in the portfolio, so we expect the new transaction to be alloacted as follows:

1. The Movements Engine allocates 4000 units to the <b>ftse_tracker</b> strategy
2. The Movements Engine allocates 2000 units to the <b>food_retail</b> strategy

In [ ]:
request = transaction_portfolios_api.upsert_transactions(
    scope=scope,
    code=portfolio_code,
    transaction_request=[
        models.TransactionRequest(
            transaction_id="trd_0021PRORATE",
            type="BuyProRated",
            instrument_identifiers={"Instrument/default/ClientInternal": "EQ_1240"},
            transaction_date="2020-01-29",
            settlement_date="2020-01-31",
            units=6000,
            transaction_price=models.TransactionPrice(price=10),
            total_consideration=models.CurrencyAndAmount(amount=60000, currency="GBP"),
            exchange_rate=None,
            counterparty_id=None,
            source="",
            properties={},
        )
    ],
)

Then check the results:

In [ ]:
response = transaction_portfolios_api.get_holdings(
    scope=scope,
    code=portfolio_code,
    property_keys=["Instrument/default/Name", "Instrument/default/ClientInternal"],
    filter="properties.Instrument/default/Name in ('Tesco', 'CCY_GBP')",
)

holdings_df = lusid_response_to_data_frame(
    response, rename_properties=True, column_name_mapping=get_holdings_json_mapping
)

holdings_df